Detecting Engine Faults using GNN

In [42]:
import pandas as pd
import torch

engine_faultdb = 'EngineFaultDB/EngineFaultDB_Final.csv'
df = pd.read_csv(engine_faultdb)

Preprocess: Normalize all columns (StandardScaler) because RPM (1000+) and Lambda (1.0) have vastly different scales. GNNs fail without this.

In [43]:
from torch.utils.data import Dataset 
from torch_geometric.data import Data

class EngineGraphDataset(Dataset):
    def __init__(self, df, labels, edge_index):
        self.x = torch.tensor(df.values, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.edge_index = edge_index

    def len(self):
        return len(self.x)
    
    def __len__(self):
        return self.len()
    
    def __getitem__(self, idx):
        x = self.x[idx].unsqueeze(1)
        y = self.labels[idx]
        return Data(x=x, edge_index=self.edge_index, y=y)

In [44]:
import numpy as np

def create_graph_nodes(df):
  corr_matrix = df.corr()

  threshold = 0.6

  mask = (corr_matrix > threshold) & (np.eye(len(corr_matrix)) == 0)

  src, dst = np.where(mask)

  edges = torch.tensor([src, dst], dtype=torch.long)
  return edges

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import DataLoader

df = pd.read_csv('EngineFaultDB/EngineFaultDB_Final.csv')
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, shuffle=True) # Shuffle=True for training!

y_train = train_data['Fault']
X_train = train_data.drop(columns=['Fault'])

y_test = test_data['Fault']
X_test = test_data.drop(columns=['Fault'])

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

edge_index = create_graph_nodes(X_train_scaled)

train_labels_binary = y_train.apply(lambda x: 0 if x == 0 else 1).values
train_dataset = EngineGraphDataset(X_train_scaled, train_labels_binary, edge_index)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_labels_binary = y_test.apply(lambda x: 0 if x == 0 else 1).values
test_dataset = EngineGraphDataset(X_test_scaled, test_labels_binary, edge_index)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_18497/1606478351.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_18497/1606478351.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Graph Construction: Create an adjacency matrix based on correlation (e.g., TPS vs RPM, AFR vs Lambda).

Compute Correlation Matrix

In [46]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F

class GNN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch) 
        x = self.lin(x)
        return x

Train: Train a GCN to classify the "Fault" label.

In [ ]:
from tqdm.auto import tqdm

model = GNN(num_node_features=1, hidden_channels=32, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

model.train()
for epoch in range(1, 21):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}", unit="batch")
    
    for batch in loop:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Avg Loss: {avg_loss:.4f}")

Epoch 0: 100%|██████████| 700/700 [00:03<00:00, 227.53batch/s, loss=0.478]


Epoch 0 finished. Avg Loss: 0.5355


Epoch 1: 100%|██████████| 700/700 [00:02<00:00, 246.30batch/s, loss=0.347]


Epoch 1 finished. Avg Loss: 0.4690


Epoch 2: 100%|██████████| 700/700 [00:02<00:00, 234.53batch/s, loss=0.425]


...

Epoch 17 finished. Avg Loss: 0.1641


Epoch 18: 100%|██████████| 700/700 [00:02<00:00, 246.18batch/s, loss=0.186] 


Epoch 18 finished. Avg Loss: 0.1587


Epoch 19: 100%|██████████| 700/700 [00:02<00:00, 244.87batch/s, loss=0.0881]

Epoch 19 finished. Avg Loss: 0.1562


In [49]:
import torch

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.batch)
      
        preds = out.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())


In [55]:
df_results = X_test_scaled.copy() 

df_results['True Label'] = all_labels 
df_results['Predicted Label'] = all_preds

df_results['Correct'] = df_results['True Label'] == df_results['Predicted Label']

label_map = {0: 'Normal', 1: 'Fault'}
df_results['True Label'] = df_results['True Label'].map(label_map)
df_results['Predicted Label'] = df_results['Predicted Label'].map(label_map)

display(df_results.head())

errors_df = df_results[df_results['Correct'] == False]
print(f"Error %: {len(errors_df) / len(df_results)}")
print(f"False positives: {len(errors_df[errors_df['True Label'] == 'Normal'])}")
print(f"False negatives: {len(errors_df[errors_df['True Label'] == 'Fault'])}")
print(f"Total Errors: {len(errors_df)}")
display(errors_df)


,MAP,TPS,Force,Power,RPM,Consumption L/H,Consumption L/100KM,Speed,CO,HC,CO2,O2,Lambda,AFR,True Label,Predicted Label,Correct
0,-0.065874,-0.252209,-0.507059,-0.425204,1.538925,0.358754,-0.766949,1.441458,-0.675423,-0.180345,0.008081,0.193343,0.288390,0.293474,Fault,Fault,True
1,-0.187737,-0.477405,-0.016363,-0.310512,-0.567025,-0.615793,-0.413498,-0.587070,-0.738269,0.249168,-0.124563,1.131652,0.500292,0.502495,Fault,Fault,True
2,-1.002546,-0.533704,-0.552541,-0.621559,-0.524697,-0.547570,-0.208032,-0.612675,2.613670,0.142669,-1.539758,-1.660933,-2.420923,-2.425867,Fault,Fault,True
3,-0.491200,-0.386885,-0.516641,-0.536255,0.299943,0.274718,-0.107999,0.332268,0.852985,0.000204,-1.205761,-1.374972,-1.300872,-1.304564,Normal,Normal,True
4,-0.491200,-0.601042,-0.551441,-0.602574,-0.431799,-0.746365,-0.723125,-0.496213,-0.690506,-0.077221,0.073927,1.328250,0.636514,0.631203,Fault,Fault,True


Error %: 0.08428571428571428
False positives: 889
False negatives: 55
Total Errors: 944


,MAP,TPS,Force,Power,RPM,Consumption L/H,Consumption L/100KM,Speed,CO,HC,CO2,O2,Lambda,AFR,True Label,Predicted Label,Correct
9,-0.260616,-0.117534,-0.482058,-0.393215,2.389000,0.808302,-0.828557,2.271676,0.021913,-0.143103,0.282914,-0.767306,-0.513809,-0.516871,Normal,Fault,False
11,-0.114858,-0.184872,-0.479572,-0.407909,2.569636,0.845802,-0.843165,2.279913,0.030963,-0.133179,0.698979,-0.736029,0.197575,0.192567,Normal,Fault,False
18,-0.819752,-0.588899,-0.515464,-0.629361,-0.975282,-0.700733,0.341040,-1.041802,0.567414,0.098968,-0.147466,-0.293683,-0.801389,-0.804147,Normal,Fault,False
40,2.051198,0.545910,-0.737601,-0.063963,-1.332813,-0.678143,0.907262,-1.341317,-0.740280,0.072409,1.864153,2.458688,2.800938,2.803793,Normal,Fault,False
45,-0.662047,-0.701497,-0.513878,-0.602834,-0.816128,-0.934317,-0.578632,-0.806348,0.354241,0.363735,-0.264842,-0.257938,-0.967883,-0.961686,Normal,Fault,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11164,-0.284510,-0.443184,-0.361981,-0.409210,0.453938,-0.249830,-0.857456,0.416228,0.262235,0.231646,-0.401304,1.368463,-0.574352,-0.572473,Normal,Fault,False
11169,-0.357389,-0.161690,-0.483333,-0.397246,2.330579,0.796104,-0.765044,2.246369,0.026941,-0.109307,0.519575,-0.731561,0.031081,0.036058,Normal,Fault,False
11184,1.771630,0.298637,1.484779,0.502995,-0.555934,0.122008,0.897417,-0.628306,-0.742794,0.242546,0.659854,0.675902,0.197575,0.191537,Normal,Fault,False
11185,-0.345442,-0.128573,-0.485214,-0.386193,2.379793,0.899567,-0.785686,2.195655,0.011858,-0.074878,0.564426,-0.709220,0.258119,0.256406,Normal,Fault,False
